In [1]:
%cd /Users/akarsh/Documents/projects/fastInfer/StoryForge

/Users/akarsh/Documents/projects/fastInfer/StoryForge


In [2]:
import json
import math
from pathlib import Path

import numpy as np
import torch

from storyforge.config import load_config
from storyforge.training import load_initial_model, train_update
from storyforge.optimization import (
    build_optimizer,
    learning_rate_for_step,
)
from storyforge.memory import MPSMemoryMonitor

In [3]:
config = load_config("configs/default.toml")

In [4]:
assert torch.backends.mps.is_available()
device = torch.device("mps")

In [5]:
profile_model = load_initial_model(config, device)
profile_optimizer = build_optimizer(profile_model, config)

In [6]:
train_tokens = np.load(
    config["encoding"]["train_path"],
    mmap_mode="r",
)

In [7]:
rng = np.random.default_rng(config["seed"])

In [8]:
tokens_per_update = (
    config["training"]["batch_size"]
    * config["model"]["max_context"]
    * config["training"]["gradient_accumulation_steps"]
)

In [9]:
total_steps = math.ceil(
    config["training"]["token_budget"] / tokens_per_update
)

In [10]:
assert 1 <= config["profiling"]["updates"] <= total_steps

In [12]:
torch.manual_seed(config["seed"])

monitor = MPSMemoryMonitor(
    config["profiling"]["sample_interval_seconds"]
)

processed_tokens = 0

torch.mps.synchronize()
monitor.start()

try:
    for step in range(1, config["profiling"]["updates"] + 1):
        lr = learning_rate_for_step(step, total_steps, config)

        for group in profile_optimizer.param_groups:
            group["lr"] = lr

        result = train_update(
            profile_model,
            profile_optimizer,
            train_tokens,
            config,
            rng,
            device,
        )

        processed_tokens += result["processed_tokens"]
        print(f"Profiling update {step} completed.")

finally:
    torch.mps.synchronize()
    measurement = monitor.stop()


Profiling update 1 completed.
Profiling update 2 completed.
Profiling update 3 completed.
Profiling update 4 completed.
Profiling update 5 completed.
Profiling update 6 completed.
Profiling update 7 completed.
Profiling update 8 completed.
Profiling update 9 completed.
Profiling update 10 completed.


In [13]:
measurement.update({
    "measurement_type": "sampled MPS memory maxima",
    "scope": "training updates only; separate profiling run",
    "updates": config["profiling"]["updates"],
    "processed_tokens": processed_tokens,
    "batch_size": config["training"]["batch_size"],
    "context": config["model"]["max_context"],
    "gradient_accumulation_steps": (
        config["training"]["gradient_accumulation_steps"]
    ),
    "parameter_dtype": str(
        next(profile_model.parameters()).dtype
    ),
})

In [14]:
output_path = Path(config["profiling"]["output_path"])
output_path.parent.mkdir(parents=True, exist_ok=True)

In [15]:
with output_path.open("w", encoding="utf-8") as file:
    json.dump(measurement, file, indent=2)

In [16]:
print(json.dumps(measurement, indent=2))

{
  "max_observed_tensor_gib": 0.9625425338745117,
  "max_observed_driver_gib": 2.09649658203125,
  "sample_count": 625,
  "requested_sample_interval_seconds": 0.01,
  "measurement_type": "sampled MPS memory maxima",
  "scope": "training updates only; separate profiling run",
  "updates": 10,
  "processed_tokens": 163840,
  "batch_size": 8,
  "context": 256,
  "gradient_accumulation_steps": 8,
  "parameter_dtype": "torch.float32"
}
